# NearMe GIS Environmental Monitoring Map

Maps all environmental and hazard data points from the medallion pipeline onto an interactive GIS map of North Carolina. Combines EPA air quality, NOAA weather stations, USGS water monitoring sites, NCDOT crash locations, NCGS landslide inventory, NCEM flood hazard zones, and IRWIN wildland fire incidents — all color-coded by source.

In [0]:
# Unified GIS Layer — Combine all environmental monitoring data with coordinates
# Pulls from gold/silver tables, fetches NOAA station coords from NWS API, and unions into one GIS table.

import requests
from pyspark.sql.functions import col, lit

# 1. EPA Air Quality (gold table has lat/lon)
epa_gis = (spark.table("workspace.default.gold_air_quality_latest")
    .select(
        lit("EPA Air Quality").alias("source_type"),
        col("latitude"),
        col("longitude"),
        col("us_aqi").alias("metric_value"),
        col("aqi_category").alias("metric_label"),
        lit("US AQI").alias("metric_unit"),
        col("measurement_time"),
    ))

# 2. NOAA Weather — silver table has no lat/lon, so fetch station coords from NWS API
noaa_obs = spark.table("workspace.default.silver_noaa_weather_observations")

headers = {"User-Agent": "nearme-hackathon/1.0"}
try:
    resp = requests.get("https://api.weather.gov/stations?state=NC", headers=headers, timeout=15)
    station_data = resp.json()
    station_rows = []
    for feat in station_data.get("features", []):
        sid = feat.get("properties", {}).get("stationIdentifier")
        geom = feat.get("geometry", {})
        coords = geom.get("coordinates", [None, None]) if geom else [None, None]
        if sid and coords[0] is not None:
            station_rows.append((sid, float(coords[1]), float(coords[0])))  # GeoJSON is (lon, lat)
    station_coords_df = spark.createDataFrame(station_rows, ["station_id", "latitude", "longitude"])
    print(f"Fetched coordinates for {len(station_rows)} NOAA stations")
except Exception as e:
    print(f"Failed to fetch NOAA station coordinates: {e}")
    station_coords_df = spark.createDataFrame([("", 0.0, 0.0)], ["station_id", "latitude", "longitude"])

noaa_gis = (noaa_obs
    .join(station_coords_df, on="station_id", how="left")
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull())
    .select(
        lit("NOAA Weather").alias("source_type"),
        col("latitude"),
        col("longitude"),
        col("temperature_f").alias("metric_value"),
        col("station_name").alias("metric_label"),
        lit("Temperature F").alias("metric_unit"),
        col("observation_time").alias("measurement_time"),
    ))

# 3. USGS Water Data (gold table has lat/lon)
usgs_gis = (spark.table("workspace.default.gold_water_conditions")
    .select(
        lit("USGS Water").alias("source_type"),
        col("latitude"),
        col("longitude"),
        col("value").alias("metric_value"),
        col("variable_name").alias("metric_label"),
        col("unit").alias("metric_unit"),
        col("observation_time").alias("measurement_time"),
    ))

# 4. NCDOT Crashes — sample 10% of 5000 for map readability (silver table has lat/lon)
ncdot_gis = (spark.table("workspace.default.silver_ncdot_crash_data")
    .sample(0.1)
    .select(
        lit("NCDOT Crash").alias("source_type"),
        col("latitude"),
        col("longitude"),
        lit(None).cast("double").alias("metric_value"),
        col("road").alias("metric_label"),
        col("condition").alias("metric_unit"),
        col("start_datetime").alias("measurement_time"),
    ))

# 5. NCGS Landslides (silver table has lat/lon)
landslide_gis = (spark.table("workspace.default.silver_ncgs_landslide_data")
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull())
    .select(
        lit("NCGS Landslide").alias("source_type"),
        col("latitude"),
        col("longitude"),
        lit(None).cast("double").alias("metric_value"),
        col("impact").alias("metric_label"),
        lit("Landslide").alias("metric_unit"),
        col("ingested_at").alias("measurement_time"),
    ))

# 6. NCEM Flood Zones (silver table has centroid lat/lon)
flood_gis = (spark.table("workspace.default.silver_ncem_flood_zones")
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull())
    .sample(0.2)  # Sample 20% of 5000 for map readability
    .select(
        lit("NCEM Flood Zone").alias("source_type"),
        col("latitude"),
        col("longitude"),
        col("depth").alias("metric_value"),
        col("zone_type").alias("metric_label"),
        lit("ft depth").alias("metric_unit"),
        lit(None).cast("timestamp").alias("measurement_time"),
    ))

# 7. IRWIN Wildland Fire Incidents (silver table has lat/lon)
fire_gis = (spark.table("workspace.default.silver_irwin_fire_incidents")
    .filter(col("latitude").isNotNull() & col("longitude").isNotNull())
    .select(
        lit("IRWIN Fire").alias("source_type"),
        col("latitude"),
        col("longitude"),
        col("incident_size").alias("metric_value"),
        col("incident_name").alias("metric_label"),
        col("fire_cause").alias("metric_unit"),
        col("fire_discovery_datetime").alias("measurement_time"),
    ))

# Union all sources into one unified GIS layer
unified_gis = (epa_gis
    .unionByName(noaa_gis)
    .unionByName(usgs_gis)
    .unionByName(ncdot_gis)
    .unionByName(landslide_gis)
    .unionByName(flood_gis)
    .unionByName(fire_gis))

# Save as Delta table for reuse
unified_gis.write.mode("overwrite").saveAsTable("workspace.default.gis_environmental_monitoring")

total = unified_gis.count()
print(f"\nUnified GIS layer: {total} total points")
print("\nPoints by source:")
display(unified_gis.groupBy("source_type").count().orderBy("count", ascending=False))

In [0]:
# Interactive Folium Map — All environmental monitoring points on NC map
# Color-coded markers with popups, clustered by source type.

%pip install folium -q

import folium
from folium.plugins import MarkerCluster

# Collect unified GIS data
gis_rows = spark.table("workspace.default.gis_environmental_monitoring").collect()

# Center map on North Carolina
nc_center = [35.5, -79.2]
m = folium.Map(location=nc_center, zoom_start=7, tiles="OpenStreetMap")

# Color scheme by source type
colors = {
    "EPA Air Quality": "green",
    "NOAA Weather": "blue",
    "USGS Water": "lightblue",
    "NCDOT Crash": "red",
    "NCGS Landslide": "purple",
    "NCEM Flood Zone": "orange",
    "IRWIN Fire": "darkred",
}

# Create a marker cluster per source type
clusters = {}
for source_type, color in colors.items():
    cluster = MarkerCluster(name=source_type)
    cluster.add_to(m)
    clusters[source_type] = cluster

for row in gis_rows:
    lat, lon = row["latitude"], row["longitude"]
    source = row["source_type"]
    if lat is None or lon is None:
        continue

    # Build popup text
    popup_parts = [f"<b>{source}</b>"]
    if row["metric_label"]:
        popup_parts.append(f"Label: {row['metric_label']}")
    if row["metric_value"] is not None:
        popup_parts.append(f"Value: {row['metric_value']} {row['metric_unit'] or ''}")
    if row["measurement_time"]:
        popup_parts.append(f"Time: {row['measurement_time']}")
    popup_html = "<br>".join(popup_parts)

    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=250),
        icon=folium.Icon(color=colors.get(source, "gray"), icon="info-sign"),
    ).add_to(clusters.get(source, m))

folium.LayerControl().add_to(m)

displayHTML(m._repr_html_())

In [0]:
# Individual Layer Views — Display each data source separately with coordinates

# EPA Air Quality (single point near Raleigh)
print("EPA Air Quality Monitoring Points:")
display(spark.table("workspace.default.gold_air_quality_latest")
    .select("latitude", "longitude", "us_aqi", "aqi_category", "pm2_5", "o3", "measurement_time"))

# NOAA Weather Stations with latest readings
print("\nNOAA Weather Stations:")
display(spark.table("workspace.default.silver_noaa_weather_observations")
    .select("station_id", "station_name", "temperature_f", "wind_speed_mph", "humidity", "observation_time")
    .limit(20))

# USGS Water Monitoring Sites
print("\nUSGS Water Monitoring Sites (latest per site/variable):")
display(spark.table("workspace.default.gold_water_conditions")
    .select("site_code", "site_name", "variable_name", "value", "unit", "latitude", "longitude", "observation_time")
    .limit(20))

# NCDOT Crash Locations (sample)
print("\nNCDOT Crash Locations (sample of 20):")
display(spark.table("workspace.default.silver_ncdot_crash_data")
    .select("object_id", "road", "county_name", "latitude", "longitude", "condition", "start_datetime")
    .limit(20))

# NCGS Landslide Points
print("\nNCGS Landslide Points (sample of 20):")
display(spark.table("workspace.default.silver_ncgs_landslide_data")
    .select("object_id", "impact", "damage", "associated_storm", "fatalities", "latitude", "longitude")
    .limit(20))

# NCEM Flood Zone Centroids
print("\nNCEM Flood Zone Centroids (sample of 20):")
display(spark.table("workspace.default.silver_ncem_flood_zones")
    .select("object_id", "zone_type", "zone_subtype", "sfha", "depth", "static_bfe", "latitude", "longitude")
    .limit(20))

# IRWIN Wildland Fire Incidents
print("\nIRWIN Wildland Fire Incidents (sample of 20):")
display(spark.table("workspace.default.silver_irwin_fire_incidents")
    .select("incident_name", "fire_cause", "incident_size", "percent_contained", "county", "latitude", "longitude", "fire_discovery_datetime")
    .limit(20))

## GIS Mapping Summary

| Layer | Source | Points | Color | Key Metric |
| --- | --- | --- | --- | --- |
| Air Quality | EPA / Open-Meteo | ~1-2 | Green | US AQI |
| Weather | NOAA NWS | ~50 | Blue | Temperature (°F) |
| Water | USGS NWIS | ~757 | Light Blue | Discharge / Gage Height |
| Crashes | NCDOT TIMS | ~500 (sampled) | Red | Road / Condition |
| Landslides | NCGS Inventory | ~6,000+ | Purple | Impact / Damage |
| Flood Zones | NCEM FRIS | ~1,000 (sampled) | Orange | Zone Type / Depth |
| Wildfires | IRWIN / WFIGS | ~197 | Dark Red | Fire Cause / Size (acres) |

**Unified GIS Table:** `workspace.default.gis_environmental_monitoring`

The interactive map above shows all environmental and hazard monitoring points across North Carolina, color-coded by data source. Click on markers for detailed readings.

**Gold Summary Tables:** `gold_air_quality_latest`, `gold_weather_summary`, `gold_water_conditions`, `gold_crash_summary`, `gold_landslide_summary`, `gold_flood_zone_summary`, `gold_fire_summary`